In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans

In [ ]:
# Grab a single frame from the match video. cv2.imread only reads still
# images — on a video file it silently returns None.
VIDEO_PATH = "../input_videos/match.mp4"  # adjust to your clip
FRAME_INDEX = 0

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_INDEX)
ret, frame = cap.read()
cap.release()
assert ret, f"Could not read frame {FRAME_INDEX} from {VIDEO_PATH}"

# OpenCV decodes BGR; matplotlib expects RGB.
image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
image.shape

In [ ]:
plt.imshow(image)
plt.show()

In [ ]:
top_half_image=image[0:int(image.shape[0]/2),:]
plt.imshow(top_half_image)
plt.show()

In [ ]:
image_2d=top_half_image.reshape(-1,3)
kmeans=KMeans(n_clusters=2,random_state=0)
kmeans.fit(image_2d)

labels=kmeans.labels_
cluster_image=labels.reshape(top_half_image.shape[0],top_half_image.shape[1])

plt.imshow(cluster_image)
plt.show()

In [ ]:
corner_clusters = [
    cluster_image[0, 0],
    cluster_image[0, -1],
    cluster_image[-1, 0],
    cluster_image[-1, -1],
]
background_cluster = max(set(corner_clusters), key=corner_clusters.count)
player_cluster = 1 - background_cluster

player_color = kmeans.cluster_centers_[player_cluster]
print("Jersey color (RGB):", player_color)

# visualize with real colors instead of the meaningless default colormap
color_map = np.array([kmeans.cluster_centers_[0], kmeans.cluster_centers_[1]], dtype=np.uint8)
recolored = color_map[labels].reshape(top_half_image.shape[0], top_half_image.shape[1], 3)
plt.imshow(recolored)
plt.show()

In [ ]:
hsv_image = cv2.cvtColor(top_half_image, cv2.COLOR_RGB2HSV)

# pitch green range (OpenCV hue is 0-179)
lower_green = np.array([35, 40, 40])
upper_green = np.array([85, 255, 255])
green_mask = cv2.inRange(hsv_image, lower_green, upper_green)
player_mask = green_mask == 0

player_pixels = top_half_image.reshape(-1, 3)[player_mask.reshape(-1)]

kmeans = KMeans(n_clusters=2, random_state=0, n_init=10)
kmeans.fit(player_pixels)

labels = kmeans.labels_
counts = np.bincount(labels)
jersey_cluster = np.argmax(counts)   # shirt body has more pixels than head/number/trim

player_color = kmeans.cluster_centers_[jersey_cluster]
print("Jersey color (RGB):", player_color)

In [ ]:
swatch = np.zeros((50, 50, 3), dtype=np.uint8)
swatch[:, :] = player_color.astype(np.uint8)
plt.imshow(swatch)
plt.show()

In [ ]:
hsv_image = cv2.cvtColor(top_half_image, cv2.COLOR_RGB2HSV)

# pitch green range (OpenCV hue is 0-179) — tune if your pitch/lighting differs
lower_green = np.array([35, 40, 40])
upper_green = np.array([85, 255, 255])
green_mask = cv2.inRange(hsv_image, lower_green, upper_green)
player_mask = green_mask == 0

player_pixels = top_half_image.reshape(-1, 3)[player_mask.reshape(-1)]

kmeans = KMeans(n_clusters=2, random_state=0, n_init=10)
kmeans.fit(player_pixels)

labels = kmeans.labels_
counts = np.bincount(labels)
jersey_cluster = np.argmax(counts)  # shirt body has more pixels than head/number/trim

player_color = kmeans.cluster_centers_[jersey_cluster]
print("Jersey color (RGB):", player_color)

# visual sanity check
swatch = np.zeros((50, 50, 3), dtype=np.uint8)
swatch[:, :] = player_color.astype(np.uint8)
plt.imshow(swatch)
plt.show()196.65284178 204.65847414  33.58166923